In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/jobs_multi_location_20260808.csv")
print(df.shape)
print(df.columns.tolist())

(1433, 19)
['adref', 'id', 'description', 'redirect_url', 'category', '__CLASS__', 'company', 'location', 'contract_time', 'salary_min', 'salary_is_predicted', 'contract_type', 'created', 'title', 'salary_max', 'search_keyword', 'search_location', 'latitude', 'longitude']


In [2]:
print("before:", len(df))
print("unique ids:", df["id"].nunique())

df = df.drop_duplicates(subset="id", keep="first")
print("after:", len(df))

before: 1433
unique ids: 1336
after: 1336


In [ ]:
import ast

<class 'dict'>
London, UK
['UK', 'London']


In [5]:
areas = df["location"].apply(ast.literal_eval).apply(lambda d: d["area"])
print(areas.apply(len).value_counts().sort_index())
print()
for a in areas.head(15):
    print(a)

location
2    646
3    248
4    392
5     48
6      2
Name: count, dtype: int64

['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London', 'Croydon']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London']
['UK', 'London', 'Harrow']
['UK', 'London']
['UK', 'London']


In [6]:
df["location_parsed"] = df["location"].apply(ast.literal_eval)
df["region"] = df["location_parsed"].apply(lambda d: d["area"][1])

print(df["region"].value_counts())

region
London                851
North West England    225
South West England    113
West Midlands          81
Scotland               66
Name: count, dtype: int64


In [7]:
c = ast.literal_eval(df["company"].iloc[0])
print(c)

{'__CLASS__': 'Adzuna::API::Response::Company', 'display_name': 'Harnham - Data & Analytics Recruitment'}


In [ ]:
df["company_name"] = df["company"].apply(
    lambda s: ast.literal_eval(s).get("display_name", "UNKNOWN")
)

print((df["company_name"] == "UNKNOWN").sum())
print(df["company_name"].nunique())
print(df["company_name"].value_counts().head(25))

In [ ]:
print(df["salary_is_predicted"].value_counts(dropna=False))
print()
print(df[["salary_min", "salary_max", "salary_is_predicted"]].head(10))

In [ ]:
has_decimal = (df["salary_min"] % 1 != 0)
print(pd.crosstab(df["salary_is_predicted"], has_decimal))

In [ ]:
df_salary = df[df["salary_is_predicted"] == 0].copy()
print(len(df_salary))
print(df_salary["salary_min"].describe())

In [ ]:
print("salary_min == 0:", (df_salary["salary_min"] == 0).sum())
print("salary_min < 10000:", (df_salary["salary_min"] < 10000).sum())
print()
print(df_salary[df_salary["salary_min"] < 10000][["title", "salary_min", "salary_max", "contract_time"]])

In [ ]:
print(df_salary.nlargest(10, "salary_min")[["title", "company_name", "salary_min", "salary_max"]])

In [ ]:
df_salary = df_salary[df_salary["salary_min"] >= 10000].copy()
print(len(df_salary))
print(df_salary["salary_min"].describe())

In [ ]:
cols = ["id", "title", "company_name", "region", "search_keyword",
        "search_location", "category", "created", "contract_time",
        "contract_type", "salary_min", "salary_max", "salary_is_predicted",
        "redirect_url"]

df[cols].to_csv("../data/processed/jobs_cleaned_20260808.csv", index=False)
df_salary[cols].to_csv("../data/processed/jobs_salary_20260808.csv", index=False)

print(len(df), len(df_salary))

In [ ]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
print("ok")